# Context-Conditioned Diffusion

This notebook implements the **context-conditioned VAE** part of the Phase 5 improvement plan. It starts from the same data pipeline as `diffusion.ipynb`, but then diverges by explicitly splitting left and right context, encoding each side into learned embeddings, and using those embeddings throughout the VAE via FiLM-style conditioning and a context-dependent prior.

The notebook currently covers the VAE side of the pipeline only. The diffusion model that would later operate in this context-conditioned latent space is **not** implemented here yet.

Several of the short cells in this notebook are **demo / smoke-test cells**. Their purpose is to verify tensor shapes, debug intermediate outputs, and visually inspect results before scaling the training setup.

In [ ]:
import itertools
import json
import math
import os
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from tqdm.auto import tqdm

try:
    from torchinfo import summary
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torchinfo'])
    from torchinfo import summary

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.data_load import DataModule
from utils.losses import masked_mae, masked_rmse, full_mae, full_rmse, psnr, masked_l1_loss

print(f'project_root: {project_root}')
print(f'python: {sys.executable}')
print(f'torch: {torch.__version__}')


In [ ]:
# create output directories, checkpoints and history will be saved here
# change this if you want to save to a different location

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    root_dir = Path('/content/drive/MyDrive/ST456_Final_Project/context_diffusion')
else:
    root_dir = project_root / 'artifacts' / 'context_diffusion'

checkpoint_dir = root_dir / 'checkpoints'
vae_checkpoint_dir = checkpoint_dir / 'vae'
diffusion_checkpoint_dir = checkpoint_dir / 'diffusion'
history_dir = root_dir / 'history'

for path in [root_dir, checkpoint_dir, vae_checkpoint_dir, diffusion_checkpoint_dir, history_dir]:
    path.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
print(f'cuda available: {torch.cuda.is_available()}')
print('output directories:')
for path in [root_dir, vae_checkpoint_dir, diffusion_checkpoint_dir, history_dir]:
    print(f'  - {path}')


## Step 0-1: Setup and Data Loading

The notebook begins the same way as the original baseline: it imports standard libraries, sets up local output folders for checkpoints and history, detects the current device, and loads batches from the shared `DataModule`.

Each batch contains three key tensors:
- `x`: the masked / corrupted spectrogram
- `y`: the clean ground-truth spectrogram
- `mask`: a binary tensor where `1` marks the missing region

For local debugging, the data setup is intentionally tiny and CPU-safe: `batch_size=2`, `max_train_samples=16`, `max_val_samples=8`, `max_test_samples=8`, and `num_workers=0`. The visualisation cell below is a **demo cell** used to confirm that the masked input, target, and mask all line up correctly.

In [ ]:
dm = DataModule(
    repo_id='han2o/grant-ortsaem-processedV3',
    variant='long_gaps',
    input_key='masked_spectrogram',
    target_key='spectrogram',
    mask_key='mask',
    batch_size=2,
    streaming=True,
    max_train_samples=16,
    max_val_samples=8,
    max_test_samples=8,
    num_workers=0,
)

train_loader, val_loader, test_loader = dm.get_dataloaders()
print('Built dataloaders successfully.')
print(f'  train_loader: {type(train_loader).__name__}')
print(f'  val_loader:   {type(val_loader).__name__}')
print(f'  test_loader:  {type(test_loader).__name__}')
print(f'  batch_size:   {dm.batch_size}')
print(f'  streaming:    {dm.streaming}')
print(f'  max samples:  train={dm.max_train_samples}, val={dm.max_val_samples}, test={dm.max_test_samples}')


In [ ]:
batch = next(iter(train_loader))
x = batch['x']
y = batch['y']
mask = batch['mask']

print('Batch keys:', list(batch.keys()))
print(f'x shape:    {tuple(x.shape)}, dtype={x.dtype}, device={x.device}, min={x.min().item():.4f}, max={x.max().item():.4f}')
print(f'y shape:    {tuple(y.shape)}, dtype={y.dtype}, device={y.device}, min={y.min().item():.4f}, max={y.max().item():.4f}')
print(f'mask shape: {tuple(mask.shape)}, dtype={mask.dtype}, unique={torch.unique(mask)}')

if 'gap_seconds' in batch:
    print('gap_seconds:', batch['gap_seconds'])

mask_cols = (mask[0, 0] > 0.5).any(dim=0)
masked_idx = torch.where(mask_cols)[0]
if len(masked_idx) > 0:
    gap_start = int(masked_idx[0].item())
    gap_end = int(masked_idx[-1].item()) + 1
    print(f'Estimated gap columns for sample 0: start={gap_start}, end={gap_end}, width={gap_end - gap_start}')
else:
    print('No masked columns detected in sample 0.')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(x[0, 0].cpu(), origin='lower', aspect='auto')
axes[0].set_title('x[0,0] masked input')
axes[1].imshow(y[0, 0].cpu(), origin='lower', aspect='auto')
axes[1].set_title('y[0,0] clean target')
axes[2].imshow(mask[0, 0].cpu(), origin='lower', aspect='auto')
axes[2].set_title('mask[0,0]')
for ax in axes:
    ax.set_xlabel('time')
    ax.set_ylabel('mel bin')
plt.tight_layout()
plt.show()


## Step 2: Context Splitting

This is where the notebook diverges from `diffusion.ipynb`. Instead of feeding the corrupted spectrogram as one monolithic input, the code explicitly splits it into a **left-context view** and a **right-context view**. Each view is padded back to the original width so it remains compatible with convolutional layers.

The demo cell that follows is a small smoke test. It uses synthetic masks to confirm that the left view contains only columns before the gap and the right view contains only columns after the gap.

In [ ]:
def split_context(z: torch.Tensor, mask_latent: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Split a masked tensor into left-context and right-context views, padded back to full width."""
    if z.ndim != 4:
        raise ValueError(f'z must have shape (B, C, H, W), got {tuple(z.shape)}')
    if mask_latent.ndim != 4:
        raise ValueError(f'mask_latent must have shape (B, 1, H, W), got {tuple(mask_latent.shape)}')
    if z.shape[0] != mask_latent.shape[0] or z.shape[-2:] != mask_latent.shape[-2:]:
        raise ValueError(
            f'z and mask_latent must agree on batch/space dims, got z={tuple(z.shape)}, mask={tuple(mask_latent.shape)}'
        )

    z_left = torch.zeros_like(z)
    z_right = torch.zeros_like(z)

    for b in range(z.shape[0]):
        mask_cols = (mask_latent[b, 0] > 0.5).any(dim=0)
        masked_idx = torch.where(mask_cols)[0]

        if masked_idx.numel() == 0:
            # No gap detected: keep the full tensor in both views for easier debugging.
            z_left[b] = z[b]
            z_right[b] = z[b]
            continue

        gap_start = int(masked_idx[0].item())
        gap_end = int(masked_idx[-1].item()) + 1

        if gap_start > 0:
            z_left[b, :, :, :gap_start] = z[b, :, :, :gap_start]
        if gap_end < z.shape[-1]:
            z_right[b, :, :, gap_end:] = z[b, :, :, gap_end:]

    return z_left, z_right


In [ ]:
def nonzero_col_range(tensor_4d: torch.Tensor, sample_idx: int) -> tuple[int | None, int | None]:
    cols = (tensor_4d[sample_idx].abs().sum(dim=(0, 1)) > 0)
    idx = torch.where(cols)[0]
    if idx.numel() == 0:
        return None, None
    return int(idx[0].item()), int(idx[-1].item()) + 1

z_test = torch.ones(2, 1, 4, 10)
mask_test = torch.zeros(2, 1, 4, 10)
mask_test[0, :, :, 3:6] = 1
mask_test[1, :, :, 6:8] = 1

z_left_test, z_right_test = split_context(z_test, mask_test)

print('split_context smoke test')
print('  z_test shape:', tuple(z_test.shape))
print('  z_left shape:', tuple(z_left_test.shape))
print('  z_right shape:', tuple(z_right_test.shape))
for sample_idx in range(z_test.shape[0]):
    mask_cols = torch.where((mask_test[sample_idx, 0] > 0.5).any(dim=0))[0]
    gap_start = int(mask_cols[0].item())
    gap_end = int(mask_cols[-1].item()) + 1
    left_range = nonzero_col_range(z_left_test, sample_idx)
    right_range = nonzero_col_range(z_right_test, sample_idx)
    print(f'  sample {sample_idx}: gap=[{gap_start}, {gap_end}), left_nonzero={left_range}, right_nonzero={right_range}')


## Padding Helpers

In [ ]:
def padding(x, multiple=8):
    b, c, h, w = x.shape
    pad_h = (multiple - (h % multiple)) % multiple
    pad_w = (multiple - (w % multiple)) % multiple
    x_pad = F.pad(x, (0, pad_w, 0, pad_h), mode='constant', value=0.0)
    pad_info = {'orig_h': h, 'orig_w': w, 'pad_h': pad_h, 'pad_w': pad_w}
    return x_pad, pad_info

def unpadding(x, pad_info):
    return x[..., :pad_info['orig_h'], :pad_info['orig_w']]


In [ ]:
x_pad, pad_info = padding(x, multiple=16)
print(f'padding smoke test original shape: {tuple(x.shape)}')
print(f'padding smoke test padded shape:   {tuple(x_pad.shape)}')
print(f'padding smoke test pad_info:       {pad_info}')
x_unpad = unpadding(x_pad, pad_info)
assert x_unpad.shape == x.shape
print('padding smoke test unpadding shape matches original.')

## Step A: Frozen Plain VAE from Hugging Face

The old ContextVAE / FiLM training path is removed below. This notebook now reuses the
plain spectrogram VAE architecture from `diffusion.ipynb`, loads a pretrained checkpoint
from Hugging Face, freezes it, and uses posterior means for deterministic latent encoding.

In [ ]:
# Copied from diffusion.ipynb: plain VAE architecture for spectrogram latents.

class ConvGNSiLU(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, groups=8):
        super().__init__()

        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
        )
        self.norm = nn.GroupNorm(num_groups=min(groups, out_channels), num_channels=out_channels)
        self.act = nn.SiLU()

    def forward(self, x):
        x = self.conv(x)
        x = self.norm(x)
        x = self.act(x)
        return x


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_emb_dim=None, groups=8):
        super().__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.time_emb_dim = time_emb_dim

        self.norm1 = nn.GroupNorm(num_groups=min(groups, in_channels), num_channels=in_channels)
        self.act1 = nn.SiLU()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)

        if time_emb_dim is not None:
            self.time_proj = nn.Linear(time_emb_dim, out_channels)
        else:
            self.time_proj = None

        self.norm2 = nn.GroupNorm(num_groups=min(groups, out_channels), num_channels=out_channels)
        self.act2 = nn.SiLU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

        if in_channels != out_channels:
            self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        else:
            self.skip = nn.Identity()

    def forward(self, x, t_emb=None):
        residual = self.skip(x)

        h = self.norm1(x)
        h = self.act1(h)
        h = self.conv1(h)

        if self.time_proj is not None and t_emb is not None:
            t_out = self.time_proj(t_emb)
            h = h + t_out[:, :, None, None]

        h = self.norm2(h)
        h = self.act2(h)
        h = self.conv2(h)

        return h + residual


class DownSample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=4, stride=2, padding=1)

    def forward(self, x):
        return self.conv(x)


class UpSample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        x = self.conv(x)
        return x


class SelfAttention(nn.Module):
    def __init__(self, channels, num_heads=4):
        super().__init__()
        assert channels % num_heads == 0, "channels must be divisible by num_heads"

        self.channels = channels
        self.num_heads = num_heads
        self.head_dim = channels // num_heads

        self.norm = nn.GroupNorm(num_groups=min(8, channels), num_channels=channels)

        self.to_q = nn.Conv2d(channels, channels, kernel_size=1)
        self.to_k = nn.Conv2d(channels, channels, kernel_size=1)
        self.to_v = nn.Conv2d(channels, channels, kernel_size=1)
        self.proj_out = nn.Conv2d(channels, channels, kernel_size=1)

    def forward(self, x):
        b, c, h, w = x.shape
        residual = x

        x = self.norm(x)
        q = self.to_q(x)
        k = self.to_k(x)
        v = self.to_v(x)

        q = q.view(b, self.num_heads, self.head_dim, h * w)
        k = k.view(b, self.num_heads, self.head_dim, h * w)
        v = v.view(b, self.num_heads, self.head_dim, h * w)

        q = q.permute(0, 1, 3, 2)
        attn_scores = torch.matmul(q, k) / math.sqrt(self.head_dim)
        attn_weights = torch.softmax(attn_scores, dim=-1)

        v = v.permute(0, 1, 3, 2)
        out = torch.matmul(attn_weights, v)

        out = out.permute(0, 1, 3, 2).contiguous().view(b, c, h, w)
        out = self.proj_out(out)
        return out + residual


class Encoder(nn.Module):
    def __init__(self, in_channels=1, base_channels=64, latent_channels=4):
        super().__init__()

        self.in_conv = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)

        self.block1 = nn.Sequential(
            ResidualBlock(base_channels, base_channels),
            ResidualBlock(base_channels, base_channels),
        )
        self.down1 = DownSample(base_channels)

        self.block2 = nn.Sequential(
            ResidualBlock(base_channels, base_channels * 2),
            ResidualBlock(base_channels * 2, base_channels * 2),
        )
        self.down2 = DownSample(base_channels * 2)

        self.block3 = nn.Sequential(
            ResidualBlock(base_channels * 2, base_channels * 4),
            ResidualBlock(base_channels * 4, base_channels * 4),
        )
        self.down3 = DownSample(base_channels * 4)

        self.block4 = nn.Sequential(
            ResidualBlock(base_channels * 4, base_channels * 4),
            ResidualBlock(base_channels * 4, base_channels * 4),
        )
        self.down4 = DownSample(base_channels * 4)

        self.mid = nn.Sequential(
            ResidualBlock(base_channels * 4, base_channels * 4),
            SelfAttention(base_channels * 4, num_heads=4),
            ResidualBlock(base_channels * 4, base_channels * 4),
        )

        self.mu_head = nn.Conv2d(base_channels * 4, latent_channels, kernel_size=1)
        self.logvar_head = nn.Conv2d(base_channels * 4, latent_channels, kernel_size=1)

    def forward(self, x):
        x = self.in_conv(x)

        x = self.block1(x)
        x = self.down1(x)

        x = self.block2(x)
        x = self.down2(x)

        x = self.block3(x)
        x = self.down3(x)

        x = self.block4(x)
        x = self.down4(x)

        x = self.mid(x)

        mu = self.mu_head(x)
        logvar = self.logvar_head(x)
        return mu, logvar


class Decoder(nn.Module):
    def __init__(self, out_channels=1, base_channels=64, latent_channels=4):
        super().__init__()

        self.in_conv = nn.Conv2d(latent_channels, base_channels * 4, kernel_size=3, padding=1)

        self.mid = nn.Sequential(
            ResidualBlock(base_channels * 4, base_channels * 4),
            SelfAttention(base_channels * 4, num_heads=4),
            ResidualBlock(base_channels * 4, base_channels * 4),
        )

        self.block4 = nn.Sequential(
            ResidualBlock(base_channels * 4, base_channels * 4),
            ResidualBlock(base_channels * 4, base_channels * 4),
        )
        self.up4 = UpSample(base_channels * 4)

        self.block3 = nn.Sequential(
            ResidualBlock(base_channels * 4, base_channels * 4),
            ResidualBlock(base_channels * 4, base_channels * 2),
        )
        self.up3 = UpSample(base_channels * 2)

        self.block2 = nn.Sequential(
            ResidualBlock(base_channels * 2, base_channels * 2),
            ResidualBlock(base_channels * 2, base_channels),
        )
        self.up2 = UpSample(base_channels)

        self.block1 = nn.Sequential(
            ResidualBlock(base_channels, base_channels),
            ResidualBlock(base_channels, base_channels),
        )
        self.up1 = UpSample(base_channels)

        self.out_norm = nn.GroupNorm(num_groups=min(8, base_channels), num_channels=base_channels)
        self.out_act = nn.SiLU()
        self.out_conv = nn.Conv2d(base_channels, out_channels, kernel_size=3, padding=1)

    def forward(self, z):
        x = self.in_conv(z)
        x = self.mid(x)

        x = self.block4(x)
        x = self.up4(x)

        x = self.block3(x)
        x = self.up3(x)

        x = self.block2(x)
        x = self.up2(x)

        x = self.block1(x)
        x = self.up1(x)

        x = self.out_norm(x)
        x = self.out_act(x)
        x = self.out_conv(x)
        return x


class VAE(nn.Module):
    def __init__(self, in_channels=1, base_channels=64, latent_channels=4):
        super().__init__()

        self.encoder = Encoder(
            in_channels=in_channels,
            base_channels=base_channels,
            latent_channels=latent_channels,
        )
        self.decoder = Decoder(
            out_channels=in_channels,
            base_channels=base_channels,
            latent_channels=latent_channels,
        )
        self.latent_channels = latent_channels

    def encode(self, x):
        mu, logvar = self.encoder(x)
        return mu, logvar

    def reparameterise(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterise(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar, z

In [ ]:
from huggingface_hub import hf_hub_download

VAE_HF_REPO_ID = "han2o/inpaint_diffusion"
VAE_HF_CHECKPOINT_FILENAME = "vae/best_model.pt"

vae = VAE(in_channels=1, base_channels=32, latent_channels=4).to(device)

ckpt_path = hf_hub_download(
    repo_id=VAE_HF_REPO_ID,
    filename=VAE_HF_CHECKPOINT_FILENAME,
)

state_dict = torch.load(ckpt_path, map_location=device)
if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
    state_dict = state_dict["model_state_dict"]

vae.load_state_dict(state_dict)
vae.eval()
for p in vae.parameters():
    p.requires_grad = False

print("Loaded VAE checkpoint:", ckpt_path)
print("VAE frozen. Total params:", sum(p.numel() for p in vae.parameters()))

In [ ]:
def vae_encode(vae, x):
    """Encode x to latent using posterior mean (no sampling). Returns z of shape (B, C, H', W')."""
    mu, log_var = vae.encoder(x)
    return mu


def vae_decode(vae, z):
    """Decode latent z back to spectrogram."""
    return vae.decoder(z)


batch = next(iter(val_loader))
x_test = batch['x'][:2].to(device)
y_test = batch['y'][:2].to(device)

x_pad, pad_info = padding(x_test, multiple=16)
y_pad, _ = padding(y_test, multiple=16)

with torch.no_grad():
    z_masked = vae_encode(vae, x_pad)
    z_clean = vae_encode(vae, y_pad)
    x_recon = vae_decode(vae, z_clean)
    x_recon = unpadding(x_recon, pad_info)

print("x_pad shape:     ", tuple(x_pad.shape))
print("z_masked shape:  ", tuple(z_masked.shape))
print("z_clean shape:   ", tuple(z_clean.shape))
print("x_recon shape:   ", tuple(x_recon.shape))
print("Downscale factor H:", x_pad.shape[-2] // z_masked.shape[-2])
print("Downscale factor W:", x_pad.shape[-1] // z_masked.shape[-1])

## Step B: ContextTokenEncoder with Spatial Tokens

In [ ]:
class ContextTokenEncoder(nn.Module):
    def __init__(self, in_channels: int, base_channels: int = 64):
        super().__init__()
        self.in_channels = in_channels
        self.base_channels = base_channels

        self.conv_in = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1),
            nn.SiLU(),
        )
        self.block1 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, base_channels),
            nn.SiLU(),
        )
        self.down1 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.SiLU(),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(base_channels * 2, base_channels * 2, kernel_size=3, padding=1),
            nn.GroupNorm(8, base_channels * 2),
            nn.SiLU(),
        )
        self.down2 = nn.Sequential(
            nn.Conv2d(base_channels * 2, base_channels * 4, kernel_size=4, stride=2, padding=1),
            nn.SiLU(),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(base_channels * 4, base_channels * 4, kernel_size=3, padding=1),
            nn.GroupNorm(8, base_channels * 4),
            nn.SiLU(),
        )

    @property
    def token_dim(self):
        return self.base_channels * 4

    def forward(self, x):
        h = self.conv_in(x)
        h = self.block1(h)
        h = self.down1(h)
        h = self.block2(h)
        h = self.down2(h)
        h = self.block3(h)

        b, c, h_spatial, w_spatial = h.shape
        return h.permute(0, 2, 3, 1).reshape(b, h_spatial * w_spatial, c)

In [ ]:
ctx_token_enc = ContextTokenEncoder(in_channels=1, base_channels=64).to(device)
x_dummy = torch.randn(2, 1, 128, 256, device=device)

with torch.no_grad():
    tokens = ctx_token_enc(x_dummy)

print("Input shape:  ", tuple(x_dummy.shape))
print("Token shape:  ", tuple(tokens.shape))
print("Token dim D:  ", ctx_token_enc.token_dim)
summary(ctx_token_enc, input_size=(2, 1, 128, 256), device=str(device))

## Step C: Build Concatenated Context Tokens

In [ ]:
def build_context_tokens(
    ctx_encoder: ContextTokenEncoder,
    x: torch.Tensor,
    mask: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Build concatenated spatial context tokens from the masked spectrogram.

    Args:
        ctx_encoder: ContextTokenEncoder (shared weights for left and right).
        x:    (B, 1, H, W) masked spectrogram.
        mask: (B, 1, H, W) binary mask, 1=gap, 0=known.

    Returns:
        context_tokens: (B, N_left + N_right, D) — spatial tokens for cross-attention.
        tokens_left:    (B, N_left, D)
        tokens_right:   (B, N_right, D)
    """
    x_left, x_right = split_context(x, mask)
    tokens_left = ctx_encoder(x_left)
    tokens_right = ctx_encoder(x_right)
    context_tokens = torch.cat([tokens_left, tokens_right], dim=1)
    return context_tokens, tokens_left, tokens_right

In [ ]:
batch = next(iter(train_loader))
x_b = batch['x'][:2].to(device)
mask_b = batch['mask'][:2].to(device)

ctx_token_enc = ContextTokenEncoder(in_channels=1, base_channels=64).to(device)
with torch.no_grad():
    ctx_tokens, tok_l, tok_r = build_context_tokens(ctx_token_enc, x_b, mask_b)

print("x shape:             ", tuple(x_b.shape))
print("tokens_left shape:   ", tuple(tok_l.shape))
print("tokens_right shape:  ", tuple(tok_r.shape))
print("context_tokens shape:", tuple(ctx_tokens.shape))
print("token dim D:         ", ctx_token_enc.token_dim)

assert not ctx_tokens.isnan().any(), "NaN in context tokens"
assert not ctx_tokens.isinf().any(), "Inf in context tokens"
print("No NaN or Inf in context_tokens: OK")

## Step D: Full Pipeline Shape Test

In [ ]:
batch = next(iter(train_loader))
x = batch['x'][:2].to(device)
y = batch['y'][:2].to(device)
mask = batch['mask'][:2].to(device)

x_pad, _ = padding(x, multiple=16)
y_pad, _ = padding(y, multiple=16)
mask_pad, _ = padding(mask, multiple=16)

with torch.no_grad():
    z_masked = vae_encode(vae, x_pad)
    z_clean = vae_encode(vae, y_pad)
    m_latent = F.interpolate(mask_pad, size=z_clean.shape[-2:], mode='nearest')
    ctx_tokens, tok_l, tok_r = build_context_tokens(ctx_token_enc, x_pad, mask_pad)

print(f"{'Tensor':<20} {'Shape':<30} {'Notes'}")
print(f"{'x_pad':<20} {str(tuple(x_pad.shape)):<30} padded spectrogram")
print(f"{'z_masked':<20} {str(tuple(z_masked.shape)):<30} masked latent")
print(f"{'z_clean':<20} {str(tuple(z_clean.shape)):<30} clean latent (diffusion target)")
print(f"{'m_latent':<20} {str(tuple(m_latent.shape)):<30} mask at latent scale")
print(f"{'tokens_left':<20} {str(tuple(tok_l.shape)):<30} left context tokens")
print(f"{'tokens_right':<20} {str(tuple(tok_r.shape)):<30} right context tokens")
print(f"{'context_tokens':<20} {str(tuple(ctx_tokens.shape)):<30} cross-attention input (K, V)")

assert z_masked.shape == z_clean.shape
assert m_latent.shape[-2:] == z_clean.shape[-2:]
assert ctx_tokens.shape[-1] == ctx_token_enc.token_dim
print("All shape assertions passed.")

## Step E: Left vs Right Token Diversity Diagnostic

In [ ]:
batch = next(iter(val_loader))
x_b = batch['x'][:4].to(device)
mask_b = batch['mask'][:4].to(device)

x_b, pad_info = padding(x_b, multiple=16)
mask_b, _ = padding(mask_b, multiple=16)

with torch.no_grad():
    ctx_tokens, tok_l, tok_r = build_context_tokens(ctx_token_enc, x_b, mask_b)

left_mean = tok_l.mean(dim=1)
right_mean = tok_r.mean(dim=1)

cos_sim = F.cosine_similarity(left_mean, right_mean, dim=1)
print("Per-sample cosine similarity (left vs right mean token):")
for i, s in enumerate(cos_sim.tolist()):
    print(f"  sample {i}: {s:.4f}")

print(f"\ntokens_left  — mean={tok_l.mean():.4f}, std={tok_l.std():.4f}, min={tok_l.min():.4f}, max={tok_l.max():.4f}")
print(f"tokens_right — mean={tok_r.mean():.4f}, std={tok_r.std():.4f}, min={tok_r.min():.4f}, max={tok_r.max():.4f}")

left_nonzero = (tok_l.abs().sum(dim=-1) > 1e-6).float().mean()
right_nonzero = (tok_r.abs().sum(dim=-1) > 1e-6).float().mean()
print(f"\nFraction of non-zero tokens — left: {left_nonzero:.4f}, right: {right_nonzero:.4f}")